# Deployment

## Review (e.1.agent-memory.ipynb)

We built up to an agent with memory: 

* `act` - let the model call specific tools 
* `observe` - pass the tool output back to the model 
* `reason` - let the model reason about the tool output to decide what to do next (e.g., call another tool or just respond directly)
* `persist state` - use an in memory checkpointer to support long-running conversations with interruptions

## Goals

Now, we'll cover how to actually deploy our agent locally to Studio and to `LangGraph Cloud`.

## Concepts

There are a few central concepts to understand -

`LangGraph` —
- Python and JavaScript library 
- Allows creation of agent workflows 

`LangGraph CLI` —
- Bundles the graph code 
- Provides a task queue for managing asynchronous operations
- Offers persistence for maintaining state across interactions

`LangSmith Deployment` (formerly `LangGraph Cloud`) --
- Hosted service for the LangGraph API
- Allows deployment of graphs from GitHub repositories
- Also provides monitoring and tracing for deployed graphs
- Accessible via a unique URL for each deployment

`LangSmith Studio` (formerly `LangGraph Studio`) --
- Integrated Development Environment (IDE) for LangGraph applications
- Uses the API as its back-end, allowing real-time testing and exploration of graphs
- Can be run locally or with cloud-deployment. See below.

`LangGraph SDK` --
- Python library for programmatically interacting with LangGraph graphs
- Provides a consistent interface for working with graphs, whether served locally or in the cloud
- Allows creation of clients, access to assistants, thread management, and execution of runs

## Testing Locally

## Studio

To start the local development server, run the following command in your terminal in the `/studio` directory in this module:

```
langgraph dev
```

You should see the following output:
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

Open your browser and navigate to the **Studio UI** URL shown above.

In [7]:
from langgraph_sdk import get_client

In [8]:
# This is the URL of the local development server
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

In [13]:
assistants

[{'assistant_id': '228f9934-0cdd-5383-92c8-ee8422522cc2',
  'graph_id': 'router',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'router',
  'created_at': '2026-04-17T12:45:50.150670+00:00',
  'updated_at': '2026-04-17T12:45:50.150670+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '0e2452a4-8316-5b50-9415-ab365f4c881f',
  'graph_id': 'graph_with_reducers',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'graph_with_reducers',
  'created_at': '2026-04-17T12:45:47.650962+00:00',
  'updated_at': '2026-04-17T12:45:47.650962+00:00',
  'version': 1,
  'description': None},
 {'assistant_id': '28d99cab-ad6c-5342-aee5-400bd8dc9b8b',
  'graph_id': 'simple_graph',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'simple_graph',
  'created_at': '2026-04-17T12:45:38.861796+00:00',
  'updated_at': '2026-04-17T12:45:38.861796+00:00',
  'version': 1,
  'description': None}]

In [32]:

import asyncio
from py_compile import main

new_assistant = await client.assistants.create(
    "router",
    context={"model_name": "gpt-5.4-nano"},
    name="Nano"
)

print("Assistant created:")
new_assistant


Assistant created:


{'assistant_id': '41296bc7-9c8c-401c-924e-8054f9e9605d',
 'graph_id': 'router',
 'config': {'configurable': {'model_name': 'gpt-5.4-nano'}},
 'context': {'model_name': 'gpt-5.4-nano'},
 'metadata': {},
 'name': 'Nano',
 'created_at': '2026-04-18T01:53:21.368252+00:00',
 'updated_at': '2026-04-18T01:53:21.368252+00:00',
 'version': 1,
 'description': None}

In [31]:
# We create a thread for tracking the state of our run
thread = await client.threads.create()

Now, we can run our agent  [with `client.runs.stream`](https://docs.langchain.com/oss/python/langgraph/graph-api/#stream-and-astream) with:

* The `thread_id`
* The `graph_id`
* The `input` 
* The `stream_mode`

We are [streaming](https://docs.langchain.com/langsmith/streaming) the full value of the state after each step of the graph with `stream_mode="values"`.
 
The state is captured in the `chunk.data`. 

In [33]:
from langchain_core.messages import HumanMessage

# Input
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# Stream
async for chunk in client.runs.stream(
        thread['thread_id'],
        assistant_id=assistant,
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '2fa027d4-6d37-4673-b519-d355f108e891'}
{'content': '', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 18, 'prompt_tokens': 59, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DVpB9Xla7okLD3NFOC1ml80zWkXHG', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': 